In [31]:
import os, sys
import numpy as np
import pandas as pd

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import DataLoader

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

import torch.backends.cudnn as cudnn

# Disable cuDNN RNNs to avoid "RNN backward can only be called in training mode" bug
cudnn.enabled = False


# Make sure notebook is at project root
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)
print("CUDA available:", torch.cuda.is_available())


Project root: c:\Users\admin\Desktop\Forex_algo\code
CUDA available: True


In [2]:
from ai_decision_engine import (
    add_indicators,
    add_time_features,
    FEATURE_COLS_BASE,
    FEATURE_COLS_IND,
)

FEATURE_COLS_EXT = FEATURE_COLS_BASE + FEATURE_COLS_IND
FEATURE_COLS_EXT



['mid_o',
 'mid_h',
 'mid_l',
 'mid_c',
 'volume',
 'rsi',
 'macd',
 'macd_signal',
 'bb_lower',
 'bb_middle',
 'bb_upper',
 'atr14',
 'ema_5',
 'ema_20',
 'ema_50',
 'ema_200']

In [3]:
DATA_PATH = os.path.join(project_root, "Data", "EUR_USD_H1.parquet")
print("Data path:", DATA_PATH)

df_raw = pd.read_parquet(DATA_PATH)
print("Raw columns:", df_raw.columns.tolist())
df_raw.head()


Data path: c:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet
Raw columns: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c']


,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2016-01-07 00:00:00+00:00,542,1.07764,1.07832,1.07744,1.07778,1.07757,1.07823,1.07735,1.07770,1.07772,1.07840,1.07752,1.07787
1,2016-01-07 01:00:00+00:00,3167,1.07776,1.08100,1.07748,1.08029,1.07768,1.08092,1.07740,1.08020,1.07784,1.08109,1.07756,1.08038
2,2016-01-07 02:00:00+00:00,1567,1.08026,1.08176,1.07996,1.08152,1.08018,1.08169,1.07987,1.08144,1.08035,1.08184,1.08005,1.08159
3,2016-01-07 03:00:00+00:00,914,1.08156,1.08257,1.08150,1.08187,1.08147,1.08249,1.08142,1.08178,1.08164,1.08265,1.08157,1.08196
4,2016-01-07 04:00:00+00:00,649,1.08190,1.08256,1.08156,1.08236,1.08182,1.08247,1.08147,1.08228,1.08199,1.08264,1.08163,1.08245


In [4]:
df = df_raw.copy()

# Ensure datetime and sorted
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df.sort_values("time").reset_index(drop=True)

# target_return: log-return of next candle's close
df["target_return"] = np.log(df["mid_c"].shift(-1)) - np.log(df["mid_c"])

# Drop last row (no target there)
df = df.dropna(subset=["target_return"]).reset_index(drop=True)

df[["time", "mid_c", "target_return"]].head()


,time,mid_c,target_return
0,2016-01-07 00:00:00+00:00,1.07778,0.002326
1,2016-01-07 01:00:00+00:00,1.08029,0.001138
2,2016-01-07 02:00:00+00:00,1.08152,0.000324
3,2016-01-07 03:00:00+00:00,1.08187,0.000453
4,2016-01-07 04:00:00+00:00,1.08236,0.000333


In [5]:
# 1) Add your indicators (same as live system)
df = add_indicators(df)

# 2) Add time features (hour, day_of_week, session, initial time_idx)
df = add_time_features(df)

# 3) Sort and override time_idx with a simple counter: 0,1,2,...,N-1
df = df.sort_values("time").reset_index(drop=True)
df["time_idx"] = np.arange(len(df), dtype=int)

# 4) Drop any NaNs left from indicator warmup
df = df.dropna().reset_index(drop=True)

print("Columns now:", df.columns.tolist())
print("time_idx range:", df.time_idx.min(), "->", df.time_idx.max())
df[["time", "time_idx", "series_id", "hour", "day_of_week"] + FEATURE_COLS_EXT].head()


Columns now: ['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c', 'target_return', 'BB_MA', 'BB_UP', 'BB_LW', 'ATR_14', 'RSI_14', 'MACD', 'SIGNAL', 'HIST', 'rsi', 'macd', 'macd_signal', 'bb_lower', 'bb_middle', 'bb_upper', 'atr14', 'ema_5', 'ema_20', 'ema_50', 'ema_200', 'atr_pips', 'atr_pct', 'bb_width', 'series_id', 'time_idx', 'hour', 'day_of_week', 'session']
time_idx range: 0 -> 61272


,time,time_idx,series_id,hour,day_of_week,mid_o,mid_h,mid_l,mid_c,volume,...,macd,macd_signal,bb_lower,bb_middle,bb_upper,atr14,ema_5,ema_20,ema_50,ema_200
0,2016-01-19 07:00:00+00:00,0,eurusd,7,1,1.08736,1.08764,1.08595,1.08652,1298,...,-0.000565,-0.000376,1.087360,1.088880,1.090399,0.001128,1.087710,1.088764,1.089161,1.088334
1,2016-01-19 08:00:00+00:00,1,eurusd,8,1,1.08654,1.08846,1.08636,1.08846,1450,...,-0.000518,-0.000404,1.087218,1.088804,1.090390,0.001199,1.087960,1.088735,1.089133,1.088336
2,2016-01-19 09:00:00+00:00,2,eurusd,9,1,1.08844,1.08890,1.08702,1.08724,1626,...,-0.000572,-0.000438,1.087091,1.088718,1.090344,0.001283,1.087720,1.088593,1.089059,1.088323
3,2016-01-19 10:00:00+00:00,3,eurusd,10,1,1.08728,1.08834,1.08662,1.08730,1376,...,-0.000603,-0.000471,1.086948,1.088604,1.090259,0.001351,1.087580,1.088470,1.088990,1.088312
4,2016-01-19 11:00:00+00:00,4,eurusd,11,1,1.08728,1.08732,1.08599,1.08630,941,...,-0.000700,-0.000517,1.086639,1.088526,1.090413,0.001391,1.087153,1.088263,1.088885,1.088289


In [6]:
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Total df:", n)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Train range:", train_df.time.min(), "->", train_df.time.max())
print("Val   range:", val_df.time.min(),   "->", val_df.time.max())
print("Test  range:", test_df.time.min(),  "->", test_df.time.max())


Total df: 61273
Train: 42891 Val: 9191 Test: 9191
Train range: 2016-01-19 07:00:00+00:00 -> 2022-12-08 19:00:00+00:00
Val   range: 2022-12-08 20:00:00+00:00 -> 2024-05-31 20:00:00+00:00
Test  range: 2024-06-02 21:00:00+00:00 -> 2025-11-21 20:00:00+00:00


In [33]:
from pytorch_forecasting.data import TimeSeriesDataSet

MAX_ENCODER_LENGTH = 96
MAX_PREDICTION_LENGTH = 1
BATCH_SIZE = 1024

# ---- Main training dataset ----
training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="target_return",
    group_ids=["series_id"],

    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    time_varying_unknown_reals=FEATURE_COLS_EXT,
    time_varying_known_categoricals=["hour", "day_of_week"],
    static_categoricals=["series_id"],

    target_normalizer=None,
    allow_missing_timesteps=True,
    add_relative_time_idx=True,
    add_encoder_length=True,
    add_target_scales=False,
)

# ---- Validation and test MUST use the CLASS METHOD ----
validation = TimeSeriesDataSet.from_dataset(
    training,
    data=val_df,
    stop_randomization=True
)

test = TimeSeriesDataSet.from_dataset(
    training,
    data=test_df,
    stop_randomization=True
)

# ---- Dataloaders ----
from torch.utils.data import DataLoader

train_loader = training.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0   # IMPORTANT ON WINDOWS
)

val_loader = validation.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)

test_loader = test.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)

len(training), len(validation), len(test)

(42795, 9095, 9095)

In [34]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting.models import TemporalFusionTransformer as TFTModel
from pytorch_forecasting.metrics import MAE

hidden_size = 32
attention_heads = 4
dropout = 0.15
hidden_continuous_size = 16
LEARNING_RATE = 1e-3

tft = TFTModel.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=hidden_size,
    attention_head_size=attention_heads,
    dropout=dropout,
    hidden_continuous_size=hidden_continuous_size,
    loss=MAE(),
    output_size=1,
    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=4,
)

print("Model type:", type(tft))
print("Is LightningModule:", isinstance(tft, pl.LightningModule))


Model type: <class 'pytorch_forecasting.models.temporal_fusion_transformer._tft.TemporalFusionTransformer'>
Is LightningModule: True


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [35]:
CHECKPOINT_NAME = "eurusd_h1_tft_v2"

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename=CHECKPOINT_NAME + "-{epoch:02d}-{val_loss:.6f}",
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    min_delta=1e-5,
)

accelerator = "gpu" if torch.cuda.is_available() else "cpu"

trainer = pl.Trainer(
    max_epochs=80,
    accelerator=accelerator,
    devices=1,
    gradient_clip_val=0.5,
    precision="32",          # ✅ force full float32 – avoids Half overflow
    callbacks=[checkpoint_cb, earlystop_cb],
    log_every_n_steps=50,
)

trainer


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [36]:
trainer.fit(
    model=tft,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

print("Best checkpoint path:", checkpoint_cb.best_model_path)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | MAE                             | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 241    | train
3  | prescalers                         | ModuleDict                      | 576    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 2.0 K  | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 38.5 K | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 2.3 K  | train
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K  | train
8  | static_context_initial_hidden_lstm |

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.
c:\Users\admin\Desktop\Forex_algo\code\venv\Lib\site-packages\lightning\pytorch\loops\fit_loop.py:310: The number of training batches (41) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 58: 100%|██████████| 41/41 [01:03<00:00,  0.65it/s, v_num=42, train_loss_step=0.001, val_loss=0.000737, train_loss_epoch=0.000987]  
Best checkpoint path: c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_42\checkpoints\eurusd_h1_tft_v2-epoch=50-val_loss=0.000680.ckpt


In [41]:
import torch
import numpy as np
from pytorch_forecasting.models import TemporalFusionTransformer as TFTModel

ckpt_path = r"c:\Users\admin\Desktop\Forex_algo\code\lightning_logs\version_42\checkpoints\eurusd_h1_tft_v2-epoch=50-val_loss=0.000680.ckpt"

# 1) Reload best model
best_model = TFTModel.load_from_checkpoint(ckpt_path)

# 2) Predictions on test set
preds_raw = best_model.predict(test_loader)   # shape: [N, max_prediction_length]
preds = preds_raw.detach().cpu().reshape(-1)  # flatten to [N]

# 3) Collect true targets from test_loader
all_targets = []
for batch in test_loader:
    x, y_tuple = batch        # y_tuple = (y, weight)
    y = y_tuple[0]            # take only the target tensor
    all_targets.append(y.detach().cpu().reshape(-1))

targets = torch.cat(all_targets)

# 4) Convert to numpy
preds_np = preds.numpy()
targets_np = targets.numpy()

# 5) Metrics
mae = np.mean(np.abs(preds_np - targets_np))
print(f"Test MAE (log-return units): {mae:.8f}")

pred_dir = np.sign(preds_np)
true_dir = np.sign(targets_np)
dir_acc = (pred_dir == true_dir).mean()
print(f"Direction accuracy: {dir_acc:.4f} ({dir_acc*100:.2f}%)")

print("\nSample comparison (first 5):")
for i in range(5):
    print(f"{i}: pred={preds_np[i]:+0.6f}, true={targets_np[i]:+0.6f}")


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Test MAE (log-return units): 0.00070327
Direction accuracy: 0.5010 (50.10%)

Sample comparison (first 5):
0: pred=-0.000442, true=+0.000220
1: pred=-0.000327, true=-0.000018
2: pred=-0.000183, true=-0.000211
3: pred=+0.000496, true=+0.000560
4: pred=+0.000407, true=+0.000128


In [40]:
for batch in test_loader:
    print("Batch type:", type(batch))
    print("Batch length:", len(batch))

    x, y = batch
    print("y type:", type(y))
    print("y content:", y)
    break


Batch type: <class 'tuple'>
Batch length: 2
y type: <class 'tuple'>
y content: (tensor([[ 2.2037e-04],
        [-1.8362e-05],
        [-2.1119e-04],
        ...,
        [ 7.5142e-04],
        [-1.0448e-03],
        [ 1.2005e-03]]), None)
